# Programming Assignment: Handling Imbalanced Data

Welcome to the assignment on handling imbalanced data.

Class imbalance is one of the most common problems you will face in real-world machine learning. Fraud detection, medical diagnosis, defect detection — in all of these, the thing you care most about predicting is the rare one. A model that ignores it can look great on accuracy while being completely useless.

In this assignment, you will work through the full pipeline for dealing with imbalanced data: diagnosing it, fixing it, and evaluating your model properly. You will use real resampling techniques — class weights, SMOTE, Tomek Links, and NearMiss — and learn which evaluation metrics actually tell you the truth.

The assignment ends with a from-scratch implementation of SMOTE so you understand exactly what the library is doing under the hood.

**What You Will Do in This Assignment**

* Diagnose class imbalance and quantify how severe it is.
* Apply class weights to make classifiers pay attention to the minority class.
* Use SMOTE to generate synthetic minority samples and tune its parameters.
* Apply undersampling strategies (random, Tomek Links, NearMiss) and compare them.
* Evaluate imbalanced classifiers using the right metrics: precision, recall, F1, ROC-AUC, and PR-AUC.
* Implement SMOTE from scratch using NumPy and understand the interpolation step.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the 💾 icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents
- [Imports](#imports)
- [1 - The Class Imbalance Problem](#1)
    - [1.1 - Why Accuracy Lies](#1-1)
    - [1.2 - Setting Up an Imbalanced Dataset](#1-2)
- [2 - Diagnosing Imbalance](#2)
    - **[Exercise 1 - diagnose_imbalance](#ex-1)**
- [3 - Class Weights Approach](#3)
    - [3.1 - How Class Weights Work](#3-1)
    - **[Exercise 2 - train_with_class_weights](#ex-2)**
- [4 - Oversampling with SMOTE](#4)
    - [4.1 - The SMOTE Algorithm](#4-1)
    - **[Exercise 3 - apply_smote](#ex-3)**
- [5 - Undersampling Strategies](#5)
    - [5.1 - Three Undersampling Methods](#5-1)
    - **[Exercise 4 - apply_undersampling](#ex-4)**
- [6 - Evaluation for Imbalanced Data](#6)
    - [6.1 - Metrics That Tell the Truth](#6-1)
    - **[Exercise 5 - evaluate_imbalanced_classifier](#ex-5)**
- [7 - SMOTE from Scratch](#7)
    - [7.1 - The Interpolation Step](#7-1)
    - **[Exercise 6 - smote_from_scratch](#ex-6)**

<a name='imports'></a>
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, NearMiss

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

<a name='1'></a>
## 1 - The Class Imbalance Problem

Before jumping to solutions, let's understand what imbalanced data actually does to your model.

<a name='1-1'></a>
### 1.1 - Why Accuracy Lies

Accuracy counts correct predictions out of all predictions:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

When one class has 99% of the samples, predicting that class for everything gives 99% accuracy. The model is useless but the number looks great.

The fix is not to train harder — it is to use better metrics and to rethink how the model is trained.

The two metrics that tell you the truth for the minority class:

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN}$$

And the F1 score combines them into one number using the harmonic mean:

$$F1 = 2 \cdot \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

<a name='1-2'></a>
### 1.2 - Setting Up an Imbalanced Dataset

We will create a synthetic imbalanced dataset with 10% minority class, similar to what you might see in fraud detection.

In [ ]:
# Create an imbalanced binary classification dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    weights=[0.90, 0.10],   # 90% majority, 10% minority
    flip_y=0.01,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")
print(f"Training class distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")

Let's see what a naive model without any imbalance handling achieves:

In [ ]:
naive_model = LogisticRegression(random_state=42, max_iter=1000)
naive_model.fit(X_train, y_train)
y_pred_naive = naive_model.predict(X_test)

print("Naive model (no imbalance handling):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_naive):.3f}")
print(f"  F1 Score (minority): {f1_score(y_test, y_pred_naive):.3f}")
print()
print(classification_report(y_test, y_pred_naive, target_names=['Majority', 'Minority']))

<a name='2'></a>
## 2 - Diagnosing Imbalance

The first step is always to understand your class distribution. You need numbers, not just a gut feeling.

<a name='ex-1'></a>
#### **Exercise 1 - `diagnose_imbalance`**

**Your Task:**

Implement the `diagnose_imbalance` function that computes the class distribution statistics for a target array.

You need to implement:

* **Count samples per class**:
    * Use `np.unique(y, return_counts=True)` to get unique classes and their counts.

* **Find majority and minority classes**:
    * The majority class has the highest count: `classes[np.argmax(counts)]`.
    * The minority class has the lowest count: `classes[np.argmin(counts)]`.

* **Compute the imbalance ratio**:
    * `imbalance_ratio = counts.max() / counts.min()`

* **Return a dictionary** with keys: `class_counts`, `imbalance_ratio`, `majority_class`, `minority_class`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**Getting counts:**
* `classes, counts = np.unique(y, return_counts=True)`

**Building the dict:**
* `"class_counts": dict(zip(classes, counts))`

**Imbalance ratio:**
* `counts.max() / counts.min()`

</details>

In [ ]:
# GRADED FUNCTION: diagnose_imbalance
def diagnose_imbalance(y):
    """
    Compute class distribution statistics.

    Args:
        y: array of class labels

    Returns:
        dict with keys:
            class_counts    -- dict mapping class label to sample count
            imbalance_ratio -- majority_count / minority_count
            majority_class  -- label of the class with most samples
            minority_class  -- label of the class with fewest samples
    """
    ### START CODE HERE ###
    
    # Get unique classes and their counts
    
    # Find majority and minority class labels
    
    # Compute imbalance ratio
    
    ### END CODE HERE ###
    
    return {
        "class_counts":    None,
        "imbalance_ratio": None,
        "majority_class":  None,
        "minority_class":  None,
    }

In [ ]:
# Test your implementation
info = diagnose_imbalance(y_train)

print("Class counts:    ", info["class_counts"])
print("Majority class:  ", info["majority_class"])
print("Minority class:  ", info["minority_class"])
print(f"Imbalance ratio: {info['imbalance_ratio']:.1f}:1")

# Visualise
counts = info["class_counts"]
plt.figure(figsize=(6, 4))
plt.bar([str(k) for k in counts.keys()], list(counts.values()), color=['steelblue', 'coral'])
plt.title("Class Distribution in Training Set")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
for i, (k, v) in enumerate(counts.items()):
    plt.text(i, v + 5, str(v), ha='center')
plt.show()

##### **Expected Output**
```
Class counts:     {0: 672, 1: 78}
Majority class:   0
Minority class:   1
Imbalance ratio:  8.6:1
```


In [ ]:
unittests.exercise_1(diagnose_imbalance)

<a name='3'></a>
## 3 - Class Weights Approach

The simplest fix for imbalance. One parameter change that tells the model: *pay more attention to mistakes on the minority class*.

<a name='3-1'></a>
### 3.1 - How Class Weights Work

Setting `class_weight='balanced'` in a sklearn classifier automatically computes weights:

$$w_c = \frac{n\_samples}{n\_classes \times n\_samples_c}$$

These weights are applied to the loss function. A minority class sample with weight 5 contributes 5× as much to the total loss as a majority class sample with weight 0.5. The model can no longer afford to ignore the minority class.

<a name='ex-2'></a>
#### **Exercise 2 - `train_with_class_weights`**

**Your Task:**

Implement the `train_with_class_weights` function that trains a `LogisticRegression` model twice: once without class weights and once with `class_weight='balanced'`. Return both F1 scores and both models.

You need to implement:

* **Train without class weights**:
    * `LogisticRegression(class_weight=None, max_iter=1000, random_state=42)`
    * Fit on `X_train`, `y_train`. Compute F1 on `X_test`, `y_test`.

* **Train with balanced class weights**:
    * `LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)`
    * Same steps as above.

* **Return a dictionary** with keys: `f1_none`, `f1_balanced`, `model_none`, `model_balanced`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

**F1 score:**
* `f1_score(y_test, model.predict(X_test))`

**Return dict:**
* `return {"f1_none": ..., "f1_balanced": ..., "model_none": ..., "model_balanced": ...}`

</details>

In [ ]:
# GRADED FUNCTION: train_with_class_weights
def train_with_class_weights(X_train, X_test, y_train, y_test):
    """
    Train LogisticRegression with and without class_weight='balanced'.

    Args:
        X_train, X_test: feature arrays
        y_train, y_test: label arrays

    Returns:
        dict with keys:
            f1_none      -- F1 score (minority class) without class weights
            f1_balanced  -- F1 score (minority class) with class_weight='balanced'
            model_none     -- fitted model without class weights
            model_balanced -- fitted model with class_weight='balanced'
    """
    ### START CODE HERE ###
    
    # Train without class weights and compute F1
    
    # Train with class_weight='balanced' and compute F1
    
    ### END CODE HERE ###
    
    return {
        "f1_none":       None,
        "f1_balanced":   None,
        "model_none":    None,
        "model_balanced": None,
    }

In [ ]:
# Test your implementation
results_cw = train_with_class_weights(X_train, X_test, y_train, y_test)

print("F1 score (minority class):")
print(f"  Without class weights:  {results_cw['f1_none']:.3f}")
print(f"  With class_weight='balanced': {results_cw['f1_balanced']:.3f}")

labels = ['No weights', 'Balanced']
scores = [results_cw['f1_none'], results_cw['f1_balanced']]
plt.figure(figsize=(6, 4))
plt.bar(labels, scores, color=['steelblue', 'coral'])
plt.ylim(0, 1)
plt.title("F1 Score: No Weights vs. Balanced Class Weights")
plt.ylabel("F1 Score (minority class)")
for i, s in enumerate(scores):
    plt.text(i, s + 0.02, f"{s:.3f}", ha='center')
plt.show()

##### **Expected Output**
```
F1 score (minority class):
  Without class weights:  0.333
  With class_weight='balanced': 0.423
```


In [ ]:
unittests.exercise_2(train_with_class_weights)

<a name='4'></a>
## 4 - Oversampling with SMOTE

Class weights adjust the loss but don't change the data. SMOTE goes further by generating new synthetic minority samples.

<a name='4-1'></a>
### 4.1 - The SMOTE Algorithm

For each minority sample $x_i$, SMOTE finds its $k$ nearest minority neighbours, picks one ($x_{nn}$), then creates a new point:

$$x_{new} = x_i + \lambda \cdot (x_{nn} - x_i), \quad \lambda \sim \text{Uniform}(0, 1)$$

The new point lands somewhere on the line between two real minority samples. It is plausible but not a duplicate.

**Key rule**: Apply SMOTE only to the training set. The test set must stay untouched.

<a name='ex-3'></a>
#### **Exercise 3 - `apply_smote`**

**Your Task:**

Implement the `apply_smote` function that applies SMOTE oversampling to a training set and returns the resampled data.

You need to implement:

* **Create a SMOTE object**: `SMOTE(k_neighbors=k_neighbors, random_state=random_state)`
* **Resample**: `smote.fit_resample(X_train, y_train)`
* **Return** the resampled `X` and `y`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

```python
from imblearn.over_sampling import SMOTE   # already imported
smote = SMOTE(k_neighbors=k_neighbors, random_state=random_state)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
```

</details>

In [ ]:
# GRADED FUNCTION: apply_smote
def apply_smote(X_train, y_train, k_neighbors=5, random_state=42):
    """
    Apply SMOTE oversampling to the training data.

    Args:
        X_train:      feature array for training
        y_train:      label array for training
        k_neighbors:  number of nearest neighbours for SMOTE
        random_state: random seed

    Returns:
        X_resampled, y_resampled: resampled arrays
    """
    ### START CODE HERE ###
    
    # Create SMOTE object
    
    # Fit and resample
    
    ### END CODE HERE ###
    
    return X_resampled, y_resampled

In [ ]:
# Test your implementation
X_smote, y_smote = apply_smote(X_train, y_train, k_neighbors=5)

print("Before SMOTE:", {int(k): int(v) for k, v in zip(*np.unique(y_train, return_counts=True))})
print("After SMOTE: ", {int(k): int(v) for k, v in zip(*np.unique(y_smote, return_counts=True))})

# Train a model on the SMOTE-resampled data
model_smote = LogisticRegression(random_state=42, max_iter=1000)
model_smote.fit(X_smote, y_smote)
y_pred_smote = model_smote.predict(X_test)

print(f"\nF1 (minority) after SMOTE: {f1_score(y_test, y_pred_smote):.3f}")


##### **Expected Output**
```
Before SMOTE: {0: 672, 1: 78}
After SMOTE:  {0: 672, 1: 672}

F1 (minority) after SMOTE: 0.455
```


In [ ]:
unittests.exercise_3(apply_smote)

### 4.2 - Tuning k_neighbors

Let's see how different `k_neighbors` values affect the F1 score:

In [ ]:
k_values = [1, 3, 5, 7, 10]
f1_scores = []

for k in k_values:
    X_res, y_res = apply_smote(X_train, y_train, k_neighbors=k)
    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_res, y_res)
    f1_scores.append(f1_score(y_test, clf.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(k_values, f1_scores, marker='o', color='coral')
plt.title("SMOTE: Effect of k_neighbors on F1 Score")
plt.xlabel("k_neighbors")
plt.ylabel("F1 Score (minority)")
plt.xticks(k_values)
plt.show()

<a name='5'></a>
## 5 - Undersampling Strategies

Instead of creating more minority samples, undersampling removes majority samples. Three strategies, each with a different logic.

<a name='5-1'></a>
### 5.1 - Three Undersampling Methods

| Method | What it removes |
|:-------|:----------------|
| **Random** | Random majority samples |
| **Tomek Links** | Majority samples that form Tomek links with minority samples (borderline pairs) |
| **NearMiss-1** | Keeps majority samples closest to the minority class |

All three return a resampled training set. The test set is never touched.

<a name='ex-4'></a>
#### **Exercise 4 - `apply_undersampling`**

**Your Task:**

Implement the `apply_undersampling` function that applies one of three undersampling strategies based on the `strategy` argument.

You need to implement:

* **If `strategy == 'random'`**: use `RandomUnderSampler(random_state=random_state)`
* **If `strategy == 'tomek'`**: use `TomekLinks()`
* **If `strategy == 'nearmiss'`**: use `NearMiss(version=1)`
* **Raise `ValueError`** for any other strategy string.
* **Return** `X_resampled, y_resampled` from `sampler.fit_resample(X_train, y_train)`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

```python
if strategy == 'random':
    sampler = RandomUnderSampler(random_state=random_state)
elif strategy == 'tomek':
    sampler = TomekLinks()
elif strategy == 'nearmiss':
    sampler = NearMiss(version=1)
else:
    raise ValueError(f"Unknown strategy: {strategy}")
```

</details>

In [ ]:
# GRADED FUNCTION: apply_undersampling
def apply_undersampling(X_train, y_train, strategy='random', random_state=42):
    """
    Apply an undersampling strategy to the training data.

    Args:
        X_train:      feature array for training
        y_train:      label array for training
        strategy:     one of 'random', 'tomek', 'nearmiss'
        random_state: random seed (used by Random and NearMiss)

    Returns:
        X_resampled, y_resampled: resampled arrays
    """
    ### START CODE HERE ###
    
    # Select the undersampling method based on strategy
    
    # Fit and resample
    
    ### END CODE HERE ###
    
    return X_resampled, y_resampled

In [ ]:
# Test your implementation with all three strategies
for strat in ['random', 'tomek', 'nearmiss']:
    X_res, y_res = apply_undersampling(X_train, y_train, strategy=strat)
    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_res, y_res)
    f1 = f1_score(y_test, clf.predict(X_test))
    dist = {int(k): int(v) for k, v in zip(*np.unique(y_res, return_counts=True))}
    print(f"{strat:10s} | distribution: {dist} | F1: {f1:.3f}")


##### **Expected Output**
```
random     | distribution: {0: 78, 1: 78}   | F1: 0.415
tomek      | distribution: {0: 667, 1: 78}  | F1: 0.278
nearmiss   | distribution: {0: 78, 1: 78}   | F1: 0.379
```


In [ ]:
unittests.exercise_4(apply_undersampling)

<a name='6'></a>
## 6 - Evaluation for Imbalanced Data

We now have multiple models trained with different strategies. Let's build a proper evaluation function that computes metrics that actually matter.

<a name='6-1'></a>
### 6.1 - Metrics That Tell the Truth

For imbalanced classification you need:

| Metric | Formula | What it tells you |
|:-------|:--------|:------------------|
| **Precision** | $TP / (TP + FP)$ | Of all predicted positives, how many are real? |
| **Recall** | $TP / (TP + FN)$ | Of all actual positives, how many did we find? |
| **F1** | harmonic mean of Precision and Recall | Single number balancing both |
| **ROC-AUC** | Area under the TPR vs FPR curve | Overall ranking ability across thresholds |
| **PR-AUC** | Area under the Precision-Recall curve | Best summary for imbalanced data |

<a name='ex-5'></a>
#### **Exercise 5 - `evaluate_imbalanced_classifier`**

**Your Task:**

Implement the `evaluate_imbalanced_classifier` function that computes all five metrics and returns them as a dictionary.

You need to implement:

* **Precision**: `precision_score(y_true, y_pred)`
* **Recall**: `recall_score(y_true, y_pred)`
* **F1**: `f1_score(y_true, y_pred)`
* **ROC-AUC**: `roc_auc_score(y_true, y_prob)`
* **PR-AUC**: `average_precision_score(y_true, y_prob)`

Return a dict with keys: `precision`, `recall`, `f1`, `roc_auc`, `pr_auc`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

* `y_pred` comes from `model.predict(X_test)` — hard class labels.
* `y_prob` comes from `model.predict_proba(X_test)[:, 1]` — probability of the positive class.
* Precision, recall, and F1 use `y_pred`. ROC-AUC and PR-AUC use `y_prob`.

</details>

In [ ]:
# GRADED FUNCTION: evaluate_imbalanced_classifier
def evaluate_imbalanced_classifier(y_true, y_pred, y_prob):
    """
    Compute evaluation metrics for an imbalanced binary classifier.

    Args:
        y_true: true binary labels
        y_pred: predicted binary labels (from model.predict)
        y_prob: predicted probabilities for the positive class
                (from model.predict_proba[:, 1])

    Returns:
        dict with keys: precision, recall, f1, roc_auc, pr_auc
    """
    ### START CODE HERE ###
    
    # Compute all five metrics
    
    ### END CODE HERE ###
    
    return {
        "precision": None,
        "recall":    None,
        "f1":        None,
        "roc_auc":   None,
        "pr_auc":    None,
    }

In [ ]:
# Test your implementation by comparing strategies
strategy_results = {}

# Naive (no handling)
y_pred_n = naive_model.predict(X_test)
y_prob_n = naive_model.predict_proba(X_test)[:, 1]
strategy_results['naive'] = evaluate_imbalanced_classifier(y_test, y_pred_n, y_prob_n)

# Class weights
y_pred_cw = results_cw['model_balanced'].predict(X_test)
y_prob_cw = results_cw['model_balanced'].predict_proba(X_test)[:, 1]
strategy_results['class_weights'] = evaluate_imbalanced_classifier(y_test, y_pred_cw, y_prob_cw)

# SMOTE
y_pred_sm = model_smote.predict(X_test)
y_prob_sm = model_smote.predict_proba(X_test)[:, 1]
strategy_results['smote'] = evaluate_imbalanced_classifier(y_test, y_pred_sm, y_prob_sm)

# Print comparison table
metrics_df = pd.DataFrame(strategy_results).T
print(metrics_df.round(3).to_string())

##### **Expected Output**
```
              precision  recall     f1  roc_auc  pr_auc
naive              0.600   0.231  0.333    0.842   0.444
class_weights      0.282   0.846  0.423    0.849   0.316
smote              0.307   0.885  0.455    0.850   0.305
```


In [ ]:
# Plot PR curves for each strategy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

models_to_plot = {
    'Naive':         (naive_model, y_prob_n),
    'Class Weights': (results_cw['model_balanced'], y_prob_cw),
    'SMOTE':         (model_smote, y_prob_sm),
}

for name, (_, y_prob) in models_to_plot.items():
    PrecisionRecallDisplay.from_predictions(y_test, y_prob, name=name, ax=axes[0])
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name, ax=axes[1])

axes[0].set_title("Precision-Recall Curves")
axes[1].set_title("ROC Curves")
plt.tight_layout()
plt.show()

In [ ]:
unittests.exercise_5(evaluate_imbalanced_classifier)

<a name='7'></a>
## 7 - SMOTE from Scratch

Now let's build SMOTE without using any library. This will show you exactly what the algorithm does step by step.

<a name='7-1'></a>
### 7.1 - The Interpolation Step

The core of SMOTE is this line:

$$x_{new} = x_i + \lambda \cdot (x_{nn} - x_i), \quad \lambda \in [0, 1]$$

When $\lambda = 0$, you get $x_i$ itself. When $\lambda = 1$, you get $x_{nn}$. For any value in between, you get a point on the line segment between them — a new, synthetic minority sample.

One subtlety: when you fit k-NN on the minority samples and then query one of those same samples, the nearest neighbour at distance 0 is itself. You need to fit `k + 1` neighbours and skip the first result.

<a name='ex-6'></a>
#### **Exercise 6 - `smote_from_scratch`**

**Your Task:**

Implement SMOTE from scratch using NumPy and `sklearn.neighbors.NearestNeighbors`.

You need to implement:

1. **Separate classes**: use `np.unique` to find majority and minority classes. Split `X` into `X_min` and `X_maj`.

2. **Count synthetic samples needed**: `n_synthetic = len(X_maj) - len(X_min)`

3. **Fit k-NN on minority samples**: `NearestNeighbors(n_neighbors=k_neighbors + 1)` — the +1 is because the sample itself will be its own nearest neighbour.

4. **Generate synthetic samples** (loop `n_synthetic` times):
    * Pick a random minority index `idx`.
    * Get `x_i = X_min[idx]`.
    * Find its neighbours: `nn.kneighbors([x_i], return_distance=False)[0]`.
    * Skip index 0 (self), pick a random neighbour `x_nn`.
    * Draw `lam = rng.uniform(0, 1)`.
    * Create `x_new = x_i + lam * (x_nn - x_i)`.

5. **Stack and return**: `np.vstack([X, X_synthetic])` and `np.concatenate([y, y_synthetic])`.

<details>
  <summary><b><font color="green">Additional Code Hints (Click to expand if you are stuck)</font></b></summary>
  
If you need a little help:

```python
rng = np.random.default_rng(random_state)
classes, counts = np.unique(y, return_counts=True)
majority_class = classes[np.argmax(counts)]
minority_class = classes[np.argmin(counts)]
X_min = X[y == minority_class]
X_maj = X[y == majority_class]
n_synthetic = len(X_maj) - len(X_min)

nn_model = NearestNeighbors(n_neighbors=k_neighbors + 1)
nn_model.fit(X_min)
```

</details>

In [ ]:
# GRADED FUNCTION: smote_from_scratch
def smote_from_scratch(X, y, k_neighbors=5, random_state=42):
    """
    Implement SMOTE from scratch using NumPy.

    Args:
        X:            feature array
        y:            label array (binary)
        k_neighbors:  number of nearest minority neighbours to consider
        random_state: random seed

    Returns:
        X_resampled, y_resampled: arrays with synthetic minority samples added
    """
    rng = np.random.default_rng(random_state)
    
    ### START CODE HERE ###
    
    # 1. Separate majority and minority samples
    
    # 2. Compute number of synthetic samples needed
    
    # 3. Fit k-NN on minority samples (k_neighbors + 1)
    
    # 4. Generate synthetic samples
    synthetic_samples = []
    for _ in range(n_synthetic):
        # Pick a random minority sample
        
        # Get its nearest neighbours (skip index 0 — that's the sample itself)
        
        # Pick one random neighbour
        
        # Interpolate
        
        pass  # remove this once you fill in the loop
    
    # 5. Stack synthetic samples with original data
    
    ### END CODE HERE ###
    
    return X_resampled, y_resampled

In [ ]:
# Test your implementation
X_scratch, y_scratch = smote_from_scratch(X_train, y_train, k_neighbors=5)

print("Before (from scratch SMOTE):", {int(k): int(v) for k, v in zip(*np.unique(y_train, return_counts=True))})
print("After  (from scratch SMOTE):", {int(k): int(v) for k, v in zip(*np.unique(y_scratch, return_counts=True))})

# Train a model on the from-scratch SMOTE data
model_scratch = LogisticRegression(random_state=42, max_iter=1000)
model_scratch.fit(X_scratch, y_scratch)
y_pred_scratch = model_scratch.predict(X_test)

print(f"\nF1 (minority) with from-scratch SMOTE: {f1_score(y_test, y_pred_scratch):.3f}")
print("(Should be close to the imblearn SMOTE result)")


##### **Expected Output**
```
Before (from scratch SMOTE): {0: 672, 1: 78}
After  (from scratch SMOTE): {0: 672, 1: 672}

F1 (minority) with from-scratch SMOTE: 0.465
(Should be close to the imblearn SMOTE result)
```


In [ ]:
unittests.exercise_6(smote_from_scratch)

## Summary

Congratulations on finishing the assignment! Here is what you covered:

| Exercise | What you built | Key takeaway |
|:---------|:--------------|:-------------|
| 1 | `diagnose_imbalance` | Always measure the imbalance ratio first |
| 2 | `train_with_class_weights` | One parameter change can significantly improve F1 |
| 3 | `apply_smote` | SMOTE creates synthetic samples, not duplicates |
| 4 | `apply_undersampling` | Three strategies with different trade-offs |
| 5 | `evaluate_imbalanced_classifier` | Accuracy lies — use F1 and PR-AUC |
| 6 | `smote_from_scratch` | SMOTE is just interpolation between minority neighbours |

**The golden rules for imbalanced data:**
1. Never use accuracy as your main metric.
2. Start simple — try `class_weight='balanced'` first.
3. Apply SMOTE only to the training set.
4. Use PR-AUC or F1 score to compare strategies.

# Programming Assignment: Handling Imbalanced Data

Welcome to the assignment on handling imbalanced data.

Class imbalance is one of the most common problems you will face in real-world machine learning. Fraud detection, medical diagnosis, defect detection — in all of these, the thing you care most about predicting is the rare one. A model that ignores it can look great on accuracy while being completely useless.

In this assignment, you will work through the full pipeline for dealing with imbalanced data: diagnosing it, fixing it, and evaluating your model properly. You will use real resampling techniques — class weights, SMOTE, Tomek Links, and NearMiss — and learn which evaluation metrics actually tell you the truth.

The assignment ends with a from-scratch implementation of SMOTE so you understand exactly what the library is doing under the hood.

**What You Will Do in This Assignment**

* Diagnose class imbalance and quantify how severe it is.
* Apply class weights to make classifiers pay attention to the minority class.
* Use SMOTE to generate synthetic minority samples and tune its parameters.
* Apply undersampling strategies (random, Tomek Links, NearMiss) and compare them.
* Evaluate imbalanced classifiers using the right metrics: precision, recall, F1, ROC-AUC, and PR-AUC.
* Implement SMOTE from scratch using NumPy and understand the interpolation step.

Let's get started!